In [1]:
"""
Imports
"""
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from collections import Counter

In [2]:
"""
Load + Split Dataset
"""
df = pd.read_csv("dataset.csv", sep=";")
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['Label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['Label'], random_state=42)

labels = sorted(df['Label'].unique())
label_to_idx = {label: i for i, label in enumerate(labels)}
idx_to_label = {i: label for label, i in label_to_idx.items()}

In [3]:
"""
Tokenization + Vocab
"""
def tokenize(text):
    return text.lower().split()

def build_vocab(texts, max_vocab_size=20000):
    counter = Counter()
    for text in texts:
        counter.update(tokenize(text))

    vocab = {"<PAD>": 0, "<UNK>": 1}
    for i, (word, _) in enumerate(counter.most_common(max_vocab_size - 2), start=2):
        vocab[word] = i

    return vocab

vocab = build_vocab(train_df['Text'].values)

In [4]:
"""
Load GloVe Embeddings
"""
def load_glove(path, vocab, embed_dim=100):
    embeddings = np.random.normal(scale=0.6, size=(len(vocab), embed_dim))

    with open(path, encoding="utf-8") as f:
        for line in f:
            values = line.strip().split()
            word = values[0]

            if word in vocab:
                embeddings[vocab[word]] = np.asarray(values[1:], dtype='float32')

    return torch.tensor(embeddings, dtype=torch.float)

embedding_matrix = load_glove("glove.6B.100d.txt", vocab)

In [5]:
"""
Dataset
"""
class GRUDataset(Dataset):
    def __init__(self, texts, labels, vocab, label_to_idx, max_len=120):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.label_to_idx = label_to_idx
        self.max_len = max_len

    def encode(self, text):
        tokens = tokenize(text)
        ids = [self.vocab.get(t, self.vocab["<UNK>"]) for t in tokens]

        if len(ids) < self.max_len:
            ids += [self.vocab["<PAD>"]] * (self.max_len - len(ids))
        else:
            ids = ids[:self.max_len]

        return torch.tensor(ids, dtype=torch.long)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encode(self.texts[idx]),
            "labels": torch.tensor(self.label_to_idx[self.labels[idx]], dtype=torch.long)
        }

In [6]:
"""
Gru Model
"""
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, hidden_dim * 2)
        self.context = nn.Linear(hidden_dim * 2, 1, bias=False)

    def forward(self, x):
        energy = torch.tanh(self.attn(x))
        weights = torch.softmax(self.context(energy), dim=1)
        return torch.sum(weights * x, dim=1)


class GRUClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim, num_layers, num_classes, dropout):
        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False)

        self.gru = nn.GRU(
            embedding_matrix.shape[1],
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )

        self.attention = Attention(hidden_dim)

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 6, num_classes)

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        output, _ = self.gru(x)

        avg_pool = torch.mean(output, dim=1)
        max_pool, _ = torch.max(output, dim=1)
        attn_out = self.attention(output)

        x = torch.cat((avg_pool, max_pool, attn_out), dim=1)
        x = self.dropout(x)

        return self.fc(x)

In [7]:
"""
Dataloaders
"""
train_dataset = GRUDataset(train_df['Text'].values, train_df['Label'].values, vocab, label_to_idx)
val_dataset = GRUDataset(val_df['Text'].values, val_df['Label'].values, vocab, label_to_idx)
test_dataset = GRUDataset(test_df['Text'].values, test_df['Label'].values, vocab, label_to_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

In [8]:
"""
Model + Optimizer + LR Scheduler
"""
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)

model = GRUClassifier(
    embedding_matrix=embedding_matrix,
    hidden_dim=256,
    num_layers=2,
    num_classes=len(labels),
    dropout=0.3
).to(device)

# Class weights
class_counts = [train_df[train_df['Label'] == label].shape[0] for label in labels]
weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
weights = weights / weights.sum()

criterion = nn.CrossEntropyLoss(weight=weights.to(device))
optimizer = AdamW(model.parameters(), lr=5e-4)

# Scheduler (Reduce LR on plateau)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=1
)

Using device: mps


In [9]:
"""
Train + Evaluate
"""
def train_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for i, batch in enumerate(loader):
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids)
        loss = criterion(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        if (i + 1) % 1000 == 0:
            print(f"  Batch {i+1}/{len(loader)} | Loss: {loss.item():.4f}")

    return total_loss / len(loader), 100 * correct / total


def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return f1_score(all_labels, all_preds, average='weighted')

In [ ]:
"""
Training Loop With Early Stopping
"""
best_f1 = 0
patience = 3
counter = 0
epochs = 15

for epoch in range(epochs):
    print(f"\n===== Epoch {epoch+1}/{epochs} =====")

    train_loss, train_acc = train_epoch(model, train_loader)
    val_f1 = evaluate(model, val_loader)

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Acc: {train_acc:.2f}%")
    print(f"Val F1: {val_f1:.4f}")
    print("LR:", optimizer.param_groups[0]['lr'])

    # Scheduler step
    scheduler.step(val_f1)
    print("LR after scheduler step:", optimizer.param_groups[0]['lr'])

    if val_f1 > best_f1:
        best_f1 = val_f1
        counter = 0
        torch.save(model.state_dict(), "best_model_attn.pth")
        print("✓ Best model saved")
    else:
        counter += 1
        print(f"No improvement ({counter}/{patience})")

        if counter >= patience:
            print("Early stopping triggered")
            break


===== Epoch 1/15 =====
  Batch 1000/5258 | Loss: 1.1310
  Batch 2000/5258 | Loss: 0.8299
  Batch 3000/5258 | Loss: 0.6003
  Batch 4000/5258 | Loss: 0.5866
  Batch 5000/5258 | Loss: 0.3716
Train Loss: 0.7812
Train Acc: 66.72%
Val F1: 0.7343
LR: 0.0005
LR after scheduler step: 0.0005
✓ Best model saved

===== Epoch 2/15 =====
  Batch 1000/5258 | Loss: 0.5684
  Batch 2000/5258 | Loss: 0.6689
  Batch 3000/5258 | Loss: 0.6420
  Batch 4000/5258 | Loss: 0.4458
  Batch 5000/5258 | Loss: 0.3812
Train Loss: 0.5229
Train Acc: 77.85%
Val F1: 0.7909
LR: 0.0005
LR after scheduler step: 0.0005
✓ Best model saved

===== Epoch 3/15 =====
  Batch 1000/5258 | Loss: 0.4105
  Batch 2000/5258 | Loss: 0.3492
  Batch 3000/5258 | Loss: 0.2776
  Batch 4000/5258 | Loss: 0.1818
  Batch 5000/5258 | Loss: 0.1184
Train Loss: 0.4100
Train Acc: 82.70%
Val F1: 0.8117
LR: 0.0005
LR after scheduler step: 0.0005
✓ Best model saved

===== Epoch 4/15 =====
  Batch 1000/5258 | Loss: 0.3297
  Batch 2000/5258 | Loss: 0.3443
 

In [14]:
"""
Test Evaluation
"""
model.load_state_dict(torch.load("best_model_attn.pth"))
test_f1 = evaluate(model, test_loader)

print("\nFINAL TEST F1:", test_f1)


FINAL TEST F1: 0.8197514898616263


In [15]:
"""
Prediction + Submission
"""
def predict(text):
    model.eval()

    tokens = tokenize(text)
    ids = [vocab.get(t, vocab["<UNK>"]) for t in tokens]
    ids = ids[:120] + [vocab["<PAD>"]] * max(0, 120 - len(ids))

    input_tensor = torch.tensor(ids).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_tensor)
        pred = torch.argmax(logits, dim=1).item()

    return idx_to_label[pred]


subm_df = pd.read_csv("subm3.csv", sep=";")
subm_df["Label"] = subm_df["Text"].apply(predict)
subm_df = subm_df.drop(columns=["Text"])
subm_df.to_csv("output/submission.csv", sep=";", index=False)

print("Submission saved!")

Submission saved!


In [16]:
"""
Predict dataset_exemplos
"""
exemplos_df = pd.read_csv("dataset-exemplos.csv", sep=";")
real_res = exemplos_df["Label"].tolist()

predicted_res = exemplos_df["Text"].apply(predict)

# Calculate accuracy
correct = sum(1 for r, p in zip(real_res, predicted_res) if r == p)
accuracy = correct / len(real_res)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.4800


In [17]:
"""
Predict subm2_labels_revealed
"""
exemplos_df = pd.read_csv("subm2_labels_revealed.csv", sep=";")
real_res = exemplos_df["Label"].tolist()

predicted_res = exemplos_df["Text"].apply(predict)

# Calculate accuracy
correct = sum(1 for r, p in zip(real_res, predicted_res) if r == p)
accuracy = correct / len(real_res)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.3900
